<a href="https://colab.research.google.com/github/BRKodali/WeatherPredictingML/blob/main/BhargavKodaliWeatherPredictingML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [33]:
#@title Library Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import r2_score

In [ ]:
#@title Converting the CSV to a DataFrame

# Convert to DataFrame
df = pd.read_csv('WeatherDataSet.csv')

# Display the DataFrame
display(df)

In [ ]:
#@title Renaming Location Names for Readability
df['NAME'] = (df['NAME'].replace('BOULDER, CO US', 'BOULDER')
                        .replace("DULUTH INTERNATIONAL AIRPORT, MN US", "DULUTH")
                        .replace("MINNEAPOLIS ST. PAUL INTERNATIONAL AIRPORT, MN US", "MSP_INTERNATIONAL"))

In [ ]:
#@title One-Hot Encoding
df = pd.get_dummies(df, columns=['NAME'], drop_first=True)

In [ ]:
#@title Removes TAVG column
df.drop(columns = 'TAVG', inplace = True)

In [ ]:
#@title Removing STATION Column
df.drop(columns = "STATION", inplace = True)

In [23]:
#@title Forward and Backward filling
# First, fill everything you can by copying forward
df.ffill(inplace=True)

# Then, catch any blanks at the very beginning by copying backward
df.bfill(inplace=True)

In [25]:
#@title Splitting Dates into 3 different columns
# Convert the text strings into pandas datetime objects
df['DATE'] = pd.to_datetime(df['DATE'])

# Extract the time components into new numerical columns
df['Year'] = df['DATE'].dt.year
df['Month'] = df['DATE'].dt.month
df['Day'] = df['DATE'].dt.day
df['DayOfYear'] = df['DATE'].dt.dayofyear

# Drop the original text column so scikit-learn does not crash
df.drop('DATE', axis=1, inplace=True)

In [26]:
#@title Training the Model
# 1. Define your multiple targets (outputs)
# We are predicting both High and Low temperatures
y = df[['TMAX', 'TMIN']]

# 2. Define your features (X)
# Make sure you drop the targets, plus TAVG (Average Temp) so you don't leak the answer
X = df.drop(['TMAX', 'TMIN',], axis=1)

# 3. Split chronologically
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

# 4. Train a multi-output model
# HistGradientBoosting requires a wrapper to handle multiple outputs at once
base_model = HistGradientBoostingRegressor()
multi_model = MultiOutputRegressor(base_model)

# Train on the multi-column y_train
multi_model.fit(X_train, y_train)

MultiOutputRegressor(estimator=HistGradientBoostingRegressor())

In [36]:
#@title Testing The Model
# 1. Have your trained model make predictions on the blind test data
predictions = multi_model.predict(X_test)

# 2. Calculate the error and force scikit-learn to show each target separately
error_mse = mean_absolute_error(y_test, predictions, multioutput='raw_values')

print(f"TMAX MSE: {error_mse[0]} degrees")
print(f"TMIN MSE: {error_mse[1]} degrees")

# 3. Take the square root of the MSE to get the RMSE
error_rmse = np.sqrt(errors_mse)

print(f"TMAX RMSE: {error_rmse[0]} degrees")
print(f"TMIN RMSE: {error_rmse[1]} degrees")

# 4. Calculates R-squared for TMAX and TMIN independently
error_r2 = r2_score(y_test, predictions, multioutput='raw_values')

print(f"TMAX R-Squared: {error_r2[0]}")
print(f"TMIN R-Squared: {error_r2[1]}")

TMAX MSE: 4.636007269016899 degrees
TMIN MSE: 4.261484710427303 degrees
TMAX RMSE: 2.1531389339791565 degrees
TMIN RMSE: 2.0643363849981675 degrees
TMAX R-Squared: 0.7749523379205174
TMIN R-Squared: 0.7914955021897834


In [19]:
#@title Display Dataframe for Debugging
display(df)

,DATE,PRCP,SNOW,SNWD,TMAX,TMIN,NAME_DULUTH,NAME_MSP_INTERNATIONAL
0,1948-01-01,0.0,0.0,NaN,6.7,-14.4,False,False
1,1948-01-02,0.0,0.0,NaN,11.1,-3.3,False,False
2,1948-01-03,0.0,0.0,NaN,11.7,1.1,False,False
3,1948-01-04,0.0,0.0,NaN,13.9,-2.2,False,False
4,1948-01-05,0.0,0.0,NaN,12.2,-5.6,False,False
...,...,...,...,...,...,...,...,...
85868,2026-10-25,0.5,0.0,0.0,12.8,8.9,False,True
85869,2026-10-26,9.7,0.0,0.0,9.4,5.6,False,True
85870,2026-10-27,3.6,0.0,0.0,13.3,6.7,False,True
85871,2026-10-28,1.0,0.0,0.0,22.2,10.6,False,True
